# Notebook 02 - Experiments (ablation study C1-C4)
**Project:** What Makes AI-Based Sparse-View 3D Gaussian Splatting Work? - Farida Gaber & Mona Yasser

This notebook runs the ablation study: the same InstantSplat pipeline with **one part changed at a time**.

| Config | Camera poses | 3D points | Refinement | Status |
|---|---|---|---|---|
| C1 | Reference (dataset) | MASt3R | Off | switch to write in Block 2 (cell "Switches") |
| C2 | MASt3R | MASt3R | Off | ready |
| C3 | MASt3R | MASt3R | On | ready (full InstantSplat; the baseline runs from notebook 01 are reused) |
| C4 | MASt3R | Random | On | switch to write in Block 2 (cell "Switches") |

**Before the first use on an account**
- Notebook 01 must have been run once (it saves the setup, the AI model and the dataset copy to Drive).
- Mona: in Google Drive, right-click the shared folder `3dgs-project` -> *Organize -> Add shortcut* -> *My Drive*, so Colab finds it at `MyDrive/3dgs-project`.

**How to use:** choose the runs in **Settings**, then `Runtime -> Run all`. Every run is saved to Drive as soon as it finishes; finished runs are skipped, so you can restart at any time.

**Time on a free T4:** about 9-21 min per run. The MASt3R step is saved on Drive the first time and reused by every configuration (saves ~1-2 min per run).

### Settings
Change only these lines.

In [ ]:
DRIVE = "/content/drive/MyDrive/3dgs-project"
REPO, DATA, LOCAL_OUT = "/content/InstantSplat", "/content/data", "/content/out"
RES_DIR, TEST_STRIDE = "images_8", 8
ALL_SCENES = ["bicycle", "garden", "stump", "treehill", "bonsai", "counter", "kitchen", "room"]
OUTDOOR, INDOOR = ["bicycle", "garden", "stump", "treehill"], ["bonsai", "counter", "kitchen", "room"]

# What to run now (examples):
#   switch check : CONFIGS_TO_RUN = ["C2"]; SCENES = ["garden"]; VIEWS = [3]
#   pilot        : CONFIGS_TO_RUN = ["C1", "C2", "C3", "C4"]; SCENES = ["garden", "bonsai"]; VIEWS = [3, 6, 9]
#   full batch   : Farida SCENES = OUTDOOR, Mona SCENES = INDOOR; all 4 configs; VIEWS = [3, 6, 9]
CONFIGS_TO_RUN = ["C2"]
SCENES = ["garden"]
VIEWS = [3]
DRY_RUN = False      # True = only list the runs, do not start them
FORCE = False        # True = redo runs that are already finished
CPU_MAINTENANCE = False   # True = no GPU needed: only update scripts/pose errors/coverage on Drive, run nothing

# Good practice for fair comparisons (see LOG.md, 12 Nov audit):
# - run all configurations of a scene in the SAME Colab session (Colab software can change between days);
# - differences below ~0.3 dB PSNR are within run-to-run noise: repeat key runs (3x) before concluding;
# - compare camera-position error (ATE) only between runs with the same number of photos.

## Our scripts
**What:** writes our four scripts (same files as `scripts/` in the GitHub repo) to Colab and to Drive.
**Do not edit here** - the switches are written in the cell "Switches" below.

In [ ]:
SCRIPTS = {'pose_error.py': '"""Camera-pose error for one sparse-view run (tracker task F2.3).\n\nWhy our own script: InstantSplat\'s metrics.py builds the reference pose from a camera-to-world\nrotation and a world-to-camera translation (mixed), and compares it with its own world-to-camera\nposes. Its ATE / RPE numbers are therefore not reliable (RPE_r came out at 78-145 degrees).\n\nHere both sides are turned into camera centres + camera orientations, aligned with ONE Sim(3)\n(Umeyama: scale, rotation, shift) fitted on the training camera centres (plus short points along each\ncamera axis, so the fit stays stable when 3 cameras are nearly on a line - version 2), then:\n  ate      RMSE of the camera-centre error, divided by the scene size\n           (mean distance of the reference centres from their centroid) -> unit-free\n  rot_deg  mean angle between aligned and reference camera orientations (degrees)\n  rpe_t    relative-pose translation error over ALL camera pairs, divided by the scene size\n  rpe_deg  relative-pose rotation error over ALL camera pairs (degrees; needs no alignment)\nSparse views are not a video, so all pairs are used instead of "consecutive" frames.\n\nUsage:\n  python pose_error.py --images-bin <scene>/sparse/0/images.bin \\\n                       --est <model>/pose/ours_1000/pose_optimized.npy \\\n                       --names IMG_1.JPG,IMG_9.JPG,...      (training photos, same order as --est)\n  python pose_error.py --selftest\n"""\nimport argparse\nimport json\nimport struct\n\nimport numpy as np\n\n\n# ----------------------------------------------------------------- reading\ndef read_images_bin(path):\n    """COLMAP images.bin -> {name: (R_w2c 3x3, t_w2c 3)}."""\n    out = {}\n    with open(path, "rb") as f:\n        (n,) = struct.unpack("<Q", f.read(8))\n        for _ in range(n):\n            f.read(4)                                         # image_id\n            qw, qx, qy, qz, tx, ty, tz = struct.unpack("<7d", f.read(56))\n            f.read(4)                                         # camera_id\n            name = b""\n            while (c := f.read(1)) != b"\\x00":\n                name += c\n            (n_pts,) = struct.unpack("<Q", f.read(8))\n            f.read(24 * n_pts)                                # x, y, point3D_id per point\n            out[name.decode()] = (qvec_to_R(np.array([qw, qx, qy, qz])), np.array([tx, ty, tz]))\n    return out\n\n\ndef qvec_to_R(q):\n    w, x, y, z = q / np.linalg.norm(q)\n    return np.array([[1 - 2 * (y * y + z * z), 2 * (x * y - w * z), 2 * (x * z + w * y)],\n                     [2 * (x * y + w * z), 1 - 2 * (x * x + z * z), 2 * (y * z - w * x)],\n                     [2 * (x * z - w * y), 2 * (y * z + w * x), 1 - 2 * (x * x + y * y)]])\n\n\ndef w2c_to_cam(R_w2c, t_w2c):\n    """world-to-camera -> (camera-to-world rotation, camera centre)."""\n    return R_w2c.T, -R_w2c.T @ t_w2c\n\n\n# ----------------------------------------------------------------- maths\ndef umeyama(X, Y):\n    """Sim(3) with Y ~ s * R @ X + t (X, Y: N x 3)."""\n    mx, my = X.mean(0), Y.mean(0)\n    Xc, Yc = X - mx, Y - my\n    U, D, Vt = np.linalg.svd(Yc.T @ Xc / len(X))\n    S = np.eye(3)\n    if np.linalg.det(U) * np.linalg.det(Vt) < 0:\n        S[2, 2] = -1\n    R = U @ S @ Vt\n    s = np.trace(np.diag(D) @ S) / (Xc ** 2).sum(1).mean()\n    return s, R, my - s * R @ mx\n\n\ndef angle_deg(Ra, Rb):\n    """Rotation angle between Ra and Rb, via atan2 (accurate also for very small angles)."""\n    M = Ra.T @ Rb\n    c = (np.trace(M) - 1) / 2\n    s = np.linalg.norm([M[2, 1] - M[1, 2], M[0, 2] - M[2, 0], M[1, 0] - M[0, 1]]) / 2\n    return float(np.degrees(np.arctan2(s, c)))\n\n\nVERSION = 2   # v2 (12 Nov): alignment uses camera centres AND camera axes (stable for 3 cameras)\n\n\nAXIS_WEIGHT = 0.1   # tested: removes the near-line artifact (7.4 -> 0.5 deg) and keeps single-camera errors honest\n\n\ndef frame_points(rot, ctr):\n    """Camera centres plus one short point along each camera axis (0.1 x the frame\'s size).\n    The axis points make the Sim(3) fit well-posed when the centres are nearly on a line (with 3\n    cameras, centres alone leave the rotation about that line almost free), while the centres still\n    dominate the fit, so one camera\'s orientation error barely spreads to the others."""\n    size = np.linalg.norm(ctr - ctr.mean(0), axis=1).mean()\n    pts = [ctr]\n    for k in range(3):\n        pts.append(ctr + AXIS_WEIGHT * size * rot[:, :, k])\n    return np.concatenate(pts, 0)\n\n\ndef flatness(ctr):\n    """Second / first singular value of the centred camera positions: 0 = cameras on a line."""\n    sv = np.linalg.svd(ctr - ctr.mean(0), compute_uv=False)\n    return float(sv[1] / sv[0]) if sv[0] > 0 else 0.0\n\n\ndef pose_error(ref_rot, ref_ctr, est_rot, est_ctr):\n    """All inputs: camera-to-world rotations (N,3,3) and camera centres (N,3), same camera order.\n    ate / rot_deg / rpe_t depend on the Sim(3) alignment; rpe_deg does not need any alignment.\n    ate is only comparable between runs with the SAME number of cameras (the fit absorbs more of the\n    error when there are fewer cameras)."""\n    ref_rot, ref_ctr, est_rot, est_ctr = map(np.asarray, (ref_rot, ref_ctr, est_rot, est_ctr))\n    n = len(ref_ctr)\n    assert n == len(est_ctr) >= 3, "need the same number of cameras on both sides (at least 3)"\n    s, R, t = umeyama(frame_points(est_rot, est_ctr), frame_points(ref_rot, ref_ctr))\n    al_ctr = (s * (R @ est_ctr.T)).T + t\n    al_rot = R @ est_rot\n    size = np.linalg.norm(ref_ctr - ref_ctr.mean(0), axis=1).mean()\n\n    ate = np.sqrt(((al_ctr - ref_ctr) ** 2).sum(1).mean()) / size\n    rot = np.mean([angle_deg(ref_rot[i], al_rot[i]) for i in range(n)])\n    rpe_t, rpe_r = [], []\n    for i in range(n):\n        for j in range(i + 1, n):\n            # relative motion of camera j seen from camera i\n            g_t = ref_rot[i].T @ (ref_ctr[j] - ref_ctr[i])\n            e_t = al_rot[i].T @ (al_ctr[j] - al_ctr[i])\n            rpe_t.append(np.linalg.norm(g_t - e_t) / size)\n            rpe_r.append(angle_deg(ref_rot[i].T @ ref_rot[j], al_rot[i].T @ al_rot[j]))\n    return {"ate": float(ate), "rot_deg": float(rot), "rpe_t": float(np.mean(rpe_t)),\n            "rpe_deg": float(np.mean(rpe_r)), "n_cams": n, "sim3_scale": float(s),\n            "flatness": flatness(ref_ctr), "pose_error_version": VERSION}\n\n\ndef evaluate_run(images_bin, est_npy, names):\n    """est_npy: InstantSplat pose_optimized.npy / pose_org.npy = world-to-camera 4x4, training order."""\n    ref = read_images_bin(images_bin)\n    est = np.load(est_npy).reshape(-1, 4, 4)\n    assert len(est) == len(names), f"{len(est)} estimated poses but {len(names)} names"\n    ref_cam = [w2c_to_cam(*ref[nm]) for nm in names]\n    est_cam = [w2c_to_cam(P[:3, :3], P[:3, 3]) for P in est]\n    return pose_error([r for r, _ in ref_cam], [c for _, c in ref_cam],\n                      [r for r, _ in est_cam], [c for _, c in est_cam])\n\n\n# ----------------------------------------------------------------- self-test\ndef random_rot(rng):\n    q = rng.normal(size=4)\n    return qvec_to_R(q)\n\n\ndef selftest():\n    rng = np.random.default_rng(0)\n    n = 9\n    rot = np.stack([random_rot(rng) for _ in range(n)])\n    ctr = rng.normal(size=(n, 3)) * 3\n\n    # 1) identity: estimate == reference\n    e = pose_error(rot, ctr, rot, ctr)\n    assert all(e[k] < 1e-6 for k in ("ate", "rot_deg", "rpe_t", "rpe_deg")), e\n\n    # 2) same cameras in another frame (scale 0.3, rotated, shifted) -> still ~0\n    s, Rg, tg = 0.3, random_rot(rng), rng.normal(size=3)\n    e = pose_error(rot, ctr, Rg @ rot, (s * (Rg @ ctr.T)).T + tg)\n    assert all(e[k] < 1e-6 for k in ("ate", "rot_deg", "rpe_t", "rpe_deg")), e\n\n    # 3) world-to-camera round trip (the format InstantSplat saves)\n    w2c = [(r.T, -r.T @ c) for r, c in zip(rot, ctr)]\n    back = [w2c_to_cam(*p) for p in w2c]\n    assert np.allclose([b[1] for b in back], ctr) and np.allclose([b[0] for b in back], rot)\n\n    # 4) a known 10-degree error on one camera is detected\n    bad = rot.copy()\n    ax = np.array([0.0, 0.0, np.radians(10)])\n    K = np.array([[0, -ax[2], ax[1]], [ax[2], 0, -ax[0]], [-ax[1], ax[0], 0]])\n    bad[0] = rot[0] @ (np.eye(3) + np.sin(np.radians(10)) / np.radians(10) * K\n                       + (1 - np.cos(np.radians(10))) / np.radians(10) ** 2 * K @ K)\n    e = pose_error(rot, ctr, bad, ctr)\n    # the axis points spread a tiny part of one camera\'s error over the others; the mean stays ~10/n\n    assert abs(e["rot_deg"] - 10 / n) < 0.05 * 10 / n and e["rpe_deg"] > 1.0, e\n\n    # 5) 3 cameras almost on a line, PERFECT rotations, 0.5 % position noise: the reported rotation\n    #    error must stay small (centre-only alignment gave a median of ~7.6 deg here)\n    def cam(a, r=3.0):\n        c = np.array([np.cos(a), np.sin(a), 0.2]) * r\n        z = -c / np.linalg.norm(c); x = np.cross([0, 0, 1], z); x /= np.linalg.norm(x); y = np.cross(z, x)\n        return np.stack([x, y, z], 1), c\n    errs = []\n    for _ in range(200):\n        R3, C3 = map(np.array, zip(*[cam(np.radians(a)) for a in (0, 180, 358)]))\n        errs.append(pose_error(R3, C3, R3, C3 + rng.normal(scale=0.015, size=C3.shape))["rot_deg"])\n    assert np.median(errs) < 1.0, np.median(errs)\n    print("selftest OK - identity, Sim(3)-invariance, w2c round trip, a known 10-degree error and "\n          "near-collinear 3-camera stability all pass")\n\n\nif __name__ == "__main__":\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--images-bin")\n    ap.add_argument("--est")\n    ap.add_argument("--names", help="comma-separated training photo names, in the order of --est")\n    ap.add_argument("--selftest", action="store_true")\n    a = ap.parse_args()\n    if a.selftest:\n        selftest()\n    else:\n        print(json.dumps(evaluate_run(a.images_bin, a.est, a.names.split(",")), indent=1))\n', 'run_all.py': '"""Batch runner for the ablation study (tracker task F4.1).\n\nRuns every (config x scene x photo count) once, saves a small result.json + logs + test renders +\ncamera poses to Drive after EVERY run, and skips runs that are already finished, so a Colab\nsession can stop at any time and the next "Run all" continues where it stopped.\n\nConfigurations (see the proposal):\n  C1  reference poses   + MASt3R points + refinement OFF   (needs hook: use_reference_poses - Block 2, Mona)\n  C2  MASt3R poses      + MASt3R points + refinement OFF   (works now: train without --optim_pose)\n  C3  MASt3R poses      + MASt3R points + refinement ON    (works now: full InstantSplat)\n  C4  MASt3R poses      + random points + refinement ON    (needs hook: use_random_points - Block 2, Mona)\n\nUsage in Colab (after the setup cells of the notebook):\n  import run_all\n  ctx = run_all.Context(drive=DRIVE, repo=REPO, data=DATA, local_out=LOCAL_OUT, manifest=manifest,\n                        commit=COMMIT, gpu=GPU_NAME)\n  run_all.run_batch(ctx, configs=["C2", "C3"], scenes=["garden"], views=[3])\n  run_all.run_batch(ctx, ..., dry_run=True)     # only print what would run\n"""\nimport datetime\nimport json\nimport os\nimport shutil\nimport subprocess\nimport tempfile\nimport threading\nimport time\nfrom dataclasses import dataclass, field\nfrom pathlib import Path\n\nimport pose_error\n\nPOSE_KEYS = ["ate", "rot_deg", "rpe_t", "rpe_deg"]\nEXPECTED_COMMIT = "b951567"     # InstantSplat version used for ALL runs (pinned in setup)\nLLFFHOLD = 8                    # init_geo.py and metrics.py use InstantSplat\'s default llffhold = 8\n\n\n# ----------------------------------------------------------------- settings shared by all runs\n@dataclass\nclass Context:\n    drive: str                       # /content/drive/MyDrive/3dgs-project\n    repo: str                        # /content/InstantSplat\n    data: str                        # /content/data\n    local_out: str                   # /content/out (heavy files, not kept)\n    manifest: dict                   # views_manifest.json: exact train/test photos per scene\n    commit: str = ""\n    gpu: str = ""\n    res_dir: str = "images_8"\n    iters: int = 1000\n    optim_test_pose_iter: int = 500\n    test_stride: int = 8\n    seed: int = 0                    # used by the C4 switch (random points); InstantSplat itself seeds training with 0\n    env_file: str = ""               # package list of this Colab session (written once per session)\n    reuse: dict = field(default_factory=lambda: {"C3": "C3_baseline"})   # finished baseline runs count as C3\n    use_init_cache: bool = True      # reuse the saved MASt3R output (init_geo) for every config of a scene/photo count\n\n\n# ----------------------------------------------------------------- MASt3R output cache (saves GPU time)\n# init_geo.py writes everything into <scene>/sparse_<n> (poses, points, confidences, test-pose start).\n# Its output is the same for C1-C4, so we run it ONCE per scene x photo count and keep a copy on Drive.\n# The switches (C1, C4) change only the local copy, after it has been restored.\ndef init_cache_dir(ctx, scene, n_views):\n    return f"{ctx.drive}/init_cache/{scene}/{n_views}_views"\n\n\ndef save_init_cache(ctx, scene, n_views, init_seconds):\n    src = Path(ctx.data) / scene / f"sparse_{n_views}"\n    assert (src / "0" / "points3D.ply").exists(), f"no init_geo output in {src}"\n    d = init_cache_dir(ctx, scene, n_views)\n    os.makedirs(d, exist_ok=True)\n    tmp = os.path.join(tempfile.gettempdir(), f"init_{scene}_{n_views}.tar")\n    subprocess.run(["tar", "-cf", tmp, "-C", str(src.parent), src.name], check=True)\n    shutil.copy(tmp, f"{d}/sparse.tar.part")\n    os.replace(f"{d}/sparse.tar.part", f"{d}/sparse.tar")\n    os.remove(tmp)\n    json.dump({"init_geo_seconds": init_seconds, "instantsplat_commit": ctx.commit, "gpu": ctx.gpu,\n               "date": datetime.datetime.now().isoformat(timespec="seconds")}, open(f"{d}/info.json", "w"), indent=1)\n\n\ndef restore_init_cache(ctx, scene, n_views):\n    """Return the original init_geo seconds if the cache was restored, else None."""\n    d = init_cache_dir(ctx, scene, n_views)\n    if not (ctx.use_init_cache and os.path.exists(f"{d}/sparse.tar") and os.path.exists(f"{d}/info.json")):\n        return None\n    info = json.load(open(f"{d}/info.json"))\n    if ctx.commit and info.get("instantsplat_commit") and info["instantsplat_commit"] != ctx.commit:\n        raise RuntimeError(f"MASt3R cache {d} was made with InstantSplat {info[\'instantsplat_commit\']}, "\n                           f"this session uses {ctx.commit}. Do not mix versions.")\n    shutil.rmtree(f"{ctx.data}/{scene}/sparse_{n_views}", ignore_errors=True)\n    subprocess.run(["tar", "-xf", f"{d}/sparse.tar", "-C", f"{ctx.data}/{scene}"], check=True)\n    return info["init_geo_seconds"]\n\n\ndef init_train_names(ctx, scene, n_views):\n    """Training photo names written by init_geo (sparse_<n>/0/images.txt), in camera-id order."""\n    rows = {}\n    with open(f"{ctx.data}/{scene}/sparse_{n_views}/0/images.txt") as f:\n        for l in f:\n            p = l.split()\n            # image lines have exactly 10 fields (ID QW QX QY QZ TX TY TZ CAMERA_ID NAME);\n            # 2D-point lines have a multiple of 3 fields or are empty\n            if len(p) == 10 and not l.startswith("#"):\n                rows[int(p[0])] = p[9]\n    return [rows[k] for k in sorted(rows)]\n\n\ndef check_init_names(ctx, scene, n_views):\n    """The MASt3R output must belong to exactly the training photos of the manifest (same order)."""\n    got = init_train_names(ctx, scene, n_views)\n    want = ctx.manifest[scene]["train_images"][str(n_views)]\n    if got != want:\n        raise RuntimeError(f"{scene} {n_views} views: MASt3R output is for {got}, manifest says {want}")\n\n\ndef env_snapshot(ctx):\n    """Save the package list of this session once (Colab updates its software without warning)."""\n    if ctx.env_file:\n        return ctx.env_file\n    os.makedirs(f"{ctx.drive}/env", exist_ok=True)\n    stamp = datetime.datetime.now().strftime("%Y-%m-%d_%H%M")\n    ctx.env_file = f"{ctx.drive}/env/pip_freeze_{stamp}.txt"\n    with open(ctx.env_file, "w") as f:\n        f.write(subprocess.run(["pip", "freeze"], capture_output=True, text=True).stdout)\n    return ctx.env_file\n\n\ndef check_context(ctx):\n    assert ctx.test_stride == LLFFHOLD, "init_geo.py / metrics.py use llffhold = 8; keep test_stride = 8"\n    if ctx.commit and not ctx.commit.startswith(EXPECTED_COMMIT[:7]):\n        raise RuntimeError(f"InstantSplat commit is {ctx.commit}, expected {EXPECTED_COMMIT}.")\n\n\n# ----------------------------------------------------------------- the switches\ndef hook_none(ctx, scene, n_views, model):\n    return {}\n\n\ndef use_reference_poses(ctx, scene, n_views, model):\n    """C1 (Block 2, Mona): replace the MASt3R training poses written by init_geo.py\n    (<scene>/sparse_<n>/0/images.bin|txt) with the dataset\'s reference poses, and move the MASt3R\n    points into the reference frame with one Sim(3) (Umeyama) fitted on the camera centres\n    (reuse pose_error.umeyama). Return what was done, e.g. {"sim3_scale": s}."""\n    raise NotImplementedError("C1 switch is a Block 2 task (M3.1)")\n\n\ndef use_random_points(ctx, scene, n_views, model):\n    """C4 (Block 2, Mona): replace the MASt3R points written by init_geo.py with the SAME number of\n    points drawn uniformly in the bounding box of (points + camera centres), fixed seed ctx.seed,\n    random colours. Return {"n_points": N, "bbox_min": [...], "bbox_max": [...]} for the log."""\n    raise NotImplementedError("C4 switch is a Block 2 task (M2.1)")\n\n\nuse_reference_poses.not_ready = True     # remove when the switch is written (notebook 02 replaces these)\nuse_random_points.not_ready = True\n\nCONFIGS = {\n    "C1": dict(after_init=use_reference_poses, optim_pose=False, label="reference poses, refinement off"),\n    "C2": dict(after_init=hook_none, optim_pose=False, label="MASt3R poses, refinement off"),\n    "C3": dict(after_init=hook_none, optim_pose=True, label="full InstantSplat"),\n    "C4": dict(after_init=use_random_points, optim_pose=True, label="random points, refinement on"),\n}\n\n\n# ----------------------------------------------------------------- helpers\nclass VramSampler(threading.Thread):\n    """Highest GPU memory used while a run is going (nvidia-smi every 2 s)."""\n\n    def __init__(self):\n        super().__init__(daemon=True)\n        self.peak, self.stop_flag = 0, False\n\n    def run(self):\n        while not self.stop_flag:\n            r = subprocess.run(["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"],\n                               capture_output=True, text=True)\n            try:\n                self.peak = max(self.peak, int(r.stdout.split()[0]))\n            except Exception:\n                pass\n            time.sleep(2)\n\n\ndef result_dir(ctx, config, scene, n_views):\n    return f"{ctx.drive}/results/{config}/{scene}/{n_views}_views"\n\n\ndef is_done(path):\n    f = Path(path) / "result.json"\n    if not f.exists():\n        return False\n    return json.load(open(f)).get("pose_metric") == "pose_error.py"\n\n\ndef commands(ctx, cfg, src, model, n_views):\n    pose_flag = " --optim_pose" if cfg["optim_pose"] else ""\n    return [\n        ("init_geo", f"python -W ignore init_geo.py -s {src} -m {model} --n_views {n_views} --focal_avg --co_vis_dsp --conf_aware_ranking"),\n        ("train", f"python train.py -s {src} -m {model} -r 1 --n_views {n_views} --iterations {ctx.iters} --pp_optimizer{pose_flag}"),\n        ("render_test", f"python render.py -s {src} -m {model} -r 1 --n_views {n_views} --iterations {ctx.iters} --eval --optim_test_pose_iter {ctx.optim_test_pose_iter}"),\n        ("metrics", f"python metrics.py -s {src} -m {model} --n_views {n_views}"),\n    ]\n\n\n# ----------------------------------------------------------------- one run\ndef run_one(ctx, config, scene, n_views, force=False, dry_run=False):\n    cfg = CONFIGS[config]\n    drive_dir = result_dir(ctx, config, scene, n_views)\n    if not force and is_done(drive_dir):\n        print(f"skip (done): {config}/{scene}/{n_views}_views")\n        return json.load(open(f"{drive_dir}/result.json"))\n    old = ctx.reuse.get(config)\n    if not force and old and is_done(result_dir(ctx, old, scene, n_views)):\n        os.makedirs(os.path.dirname(drive_dir), exist_ok=True)\n        if not dry_run:\n            shutil.copytree(result_dir(ctx, old, scene, n_views), drive_dir, dirs_exist_ok=True)\n        print(f"reuse {old} -> {config}/{scene}/{n_views}_views (same settings)")\n        return json.load(open(f"{result_dir(ctx, old, scene, n_views)}/result.json"))\n\n    src = f"{ctx.data}/{scene}"\n    model = f"{ctx.local_out}/{config}/{scene}/{n_views}_views"\n    steps = commands(ctx, cfg, src, model, n_views)\n    print(f"=== {config}/{scene}/{n_views}_views ({cfg[\'label\']})")\n    if dry_run:\n        for name, cmd in steps:\n            print(f"   [{name}] {cmd}")\n        return None\n\n    shutil.rmtree(f"{src}/sparse_{n_views}", ignore_errors=True)   # init_geo writes here; start clean (or restore cache)\n    shutil.rmtree(model, ignore_errors=True)\n    os.makedirs(model, exist_ok=True)\n    os.makedirs(drive_dir, exist_ok=True)\n\n    sampler = VramSampler()\n    sampler.start()\n    stage_s, switch_info, t_all = {}, {}, time.time()\n    env_snapshot(ctx)\n    init_cached = restore_init_cache(ctx, scene, n_views)\n    try:\n        for name, cmd in steps:\n            if name == "init_geo" and init_cached is not None:\n                stage_s[name] = 0.0\n                print(f"   init_geo     from Drive cache (originally {init_cached:.0f} s)")\n                check_init_names(ctx, scene, n_views)\n                switch_info = cfg["after_init"](ctx, scene, n_views, model) or {}\n                continue\n            t0 = time.time()\n            log = f"{model}/{name}.log"\n            with open(log, "w") as lf:\n                rc = subprocess.run(cmd, shell=True, cwd=ctx.repo, stdout=lf, stderr=subprocess.STDOUT).returncode\n            stage_s[name] = round(time.time() - t0, 1)\n            print(f"   {name:12s} {stage_s[name]:7.1f} s")\n            if rc != 0:\n                tail = "".join(open(log).readlines()[-25:])\n                raise RuntimeError(f"Stage \'{name}\' failed for {config}/{scene}/{n_views}. Last lines:\\n{tail}")\n            if name == "init_geo":\n                check_init_names(ctx, scene, n_views)\n                if ctx.use_init_cache:\n                    save_init_cache(ctx, scene, n_views, stage_s[name])      # BEFORE any switch changes it\n                switch_info = cfg["after_init"](ctx, scene, n_views, model) or {}\n    finally:\n        sampler.stop_flag = True\n\n    if not os.path.exists(f"{model}/results.json"):           # metrics.py hides its own errors\n        tail = "".join(open(f"{model}/metrics.log").readlines()[-25:])\n        raise RuntimeError(f"metrics.py wrote no results.json for {config}/{scene}/{n_views}. Log:" + chr(10) + f"{tail}")\n    res = json.load(open(f"{model}/results.json"))\n    m = res[sorted(res)[0]]\n    renders = Path(f"{model}/test/ours_{ctx.iters}/renders")\n    pose_dir = Path(f"{model}/pose/ours_{ctx.iters}")\n    names = ctx.manifest[scene]["train_images"][str(n_views)]\n    gt_bin = f"{src}/sparse/0/images.bin"\n    pe_after = pose_error.evaluate_run(gt_bin, pose_dir / "pose_optimized.npy", names)\n    pe_before = pose_error.evaluate_run(gt_bin, pose_dir / "pose_org.npy", names)\n\n    os.makedirs(f"{drive_dir}/poses", exist_ok=True)\n    for f in ["pose_org.npy", "pose_optimized.npy"]:\n        shutil.copy(pose_dir / f, f"{drive_dir}/poses/{f}")\n    for f in ["results.json", "per_view.json", "train_time.txt"] + [p.name for p in Path(model).glob("*.log")]:\n        if os.path.exists(f"{model}/{f}"):\n            shutil.copy(f"{model}/{f}", drive_dir)\n    shutil.copytree(renders, f"{drive_dir}/test_renders", dirs_exist_ok=True)\n\n    result = dict(config=config, config_label=cfg["label"], scene=scene, n_views=n_views,\n                  psnr=m["PSNR"], ssim=m["SSIM"], lpips=m["LPIPS"],\n                  **{k: pe_after[k] for k in POSE_KEYS},\n                  **{k + "_init": pe_before[k] for k in POSE_KEYS},\n                  pose_metric="pose_error.py", pose_error_version=pe_after["pose_error_version"],\n                  flatness=pe_after["flatness"], train_images=names, switch_info=switch_info,\n                  init_from_cache=init_cached is not None,\n                  init_geo_seconds=init_cached if init_cached is not None else stage_s.get("init_geo"),\n                  n_test_images=len(list(renders.glob("*"))),\n                  stage_seconds=stage_s, total_seconds=round(time.time() - t_all, 1),\n                  total_seconds_incl_init=round(time.time() - t_all + (init_cached or 0), 1),\n                  peak_vram_mb=sampler.peak, gpu=ctx.gpu, resolution=ctx.res_dir, iterations=ctx.iters,\n                  optim_test_pose_iter=ctx.optim_test_pose_iter, test_stride=ctx.test_stride, seed=ctx.seed,\n                  instantsplat_commit=ctx.commit, env_file=ctx.env_file, date=datetime.datetime.now().isoformat(timespec="seconds"))\n    json.dump(result, open(f"{drive_dir}/result.json", "w"), indent=1)\n    print(f"   done: PSNR {result[\'psnr\']:.2f} | pose before->after: ATE {result[\'ate_init\']:.3f}->{result[\'ate\']:.3f}, "\n          f"rot {result[\'rot_deg_init\']:.2f}->{result[\'rot_deg\']:.2f} deg | {result[\'total_seconds\'] / 60:.1f} min")\n    return result\n\n\n# ----------------------------------------------------------------- many runs\ndef run_batch(ctx, configs, scenes, views, force=False, dry_run=False, stop_on_error=False):\n    """Loop config x scene x views. A failed run is reported and skipped (unless stop_on_error)."""\n    check_context(ctx)\n    t0, done, failed, unavailable = time.time(), [], [], set()\n    for config in configs:\n        if getattr(CONFIGS[config]["after_init"], "not_ready", False):   # switch not written yet: no GPU spent\n            print(f"   {config}: switch not written yet - skipped")\n            failed.append((config, "-", "-", "switch not written yet"))\n            continue\n        for scene in scenes:\n            for v in views:\n                if config in unavailable:\n                    continue\n                try:\n                    r = run_one(ctx, config, scene, v, force=force, dry_run=dry_run)\n                    if r:\n                        done.append(r)\n                except NotImplementedError as e:\n                    print(f"   not available yet: {e}")\n                    failed.append((config, scene, v, str(e)))\n                    unavailable.add(config)                # skip the rest of this config\n                except Exception as e:\n                    print(f"   FAILED: {e}")\n                    failed.append((config, scene, v, str(e)[:300]))\n                    if stop_on_error:\n                        raise\n    print(f"\\nBatch finished in {(time.time() - t0) / 60:.1f} min: {len(done)} runs ready, {len(failed)} not done.")\n    for f in failed:\n        print("   not done:", f[:3], "-", f[3][:120])\n    return done, failed\n', 'aggregate.py': '"""Collect all result.json files into tables (tracker task F4.1).\n\nOutput (written next to the results folder):\n  all_runs.csv          one row per run\n  means_by_config.csv   mean over scenes for each config x photo count\n  effects.csv           the three research questions as differences of means:\n                          RQ1 poses      = C1 - C2\n                          RQ2 3D points  = C3 - C4\n                          RQ3 refinement = C3 - C2\nUsage:\n  python aggregate.py /content/drive/MyDrive/3dgs-project/results\n  python aggregate.py --selftest\n"""\nimport csv\nimport json\nimport sys\nimport tempfile\nfrom collections import defaultdict\nfrom pathlib import Path\n\nCONFIGS = ["C1", "C2", "C3", "C4", "C5"]\nALIASES = {"C3_baseline": "C3"}      # baseline runs of notebook 01 = C3 with the same settings\n# ate is only comparable between runs with the SAME photo count (tables are always per photo count)\nMETRICS = ["psnr", "ssim", "lpips", "ate", "rot_deg", "rpe_deg", "ate_init", "rot_deg_init",\n           "total_seconds_incl_init", "total_seconds", "peak_vram_mb"]\nEFFECTS = [("RQ1 poses (C1 - C2)", "C1", "C2"),\n           ("RQ2 3D points (C3 - C4)", "C3", "C4"),\n           ("RQ3 refinement (C3 - C2)", "C3", "C2")]\n\n\ndef load(results_root):\n    """One row per (config, scene, photos). Old-format runs are skipped; a C3_baseline run counts as C3\n    unless a separate C3 run exists for the same scene and photo count."""\n    rows, aliased = {}, {}\n    for f in sorted(Path(results_root).glob("*/*/*_views/result.json")):\n        r = json.load(open(f))\n        if r.get("pose_metric") != "pose_error.py":\n            continue                                   # old runs without a valid pose error\n        c = f.parts[-4]                                # config = FOLDER name (a renamed/old folder is never used)\n        if r.get("config") not in (c, None) and not (c in CONFIGS and r.get("config") in ALIASES):\n            print(f"WARNING: {f} says config {r.get(\'config\')} but is in folder {c} - folder name used")\n        if r.get("pose_error_version", 1) < 2:\n            print(f"WARNING: {f} has pose error version 1 - run recompute_pose_error first")\n        if c in ALIASES:\n            r = dict(r, config=ALIASES[c], source_config=c)\n            target = aliased\n        elif c in CONFIGS:\n            target = rows\n        else:\n            continue\n        # time of the full pipeline: runs that reused the saved MASt3R output add its original time back\n        if r.get("total_seconds") is not None:\n            extra = r.get("init_geo_seconds") if r.get("init_from_cache") else 0\n            r["total_seconds_incl_init"] = r["total_seconds"] + (extra or 0)\n        target[(r["config"], r["scene"], r["n_views"])] = r\n    for key, r in aliased.items():\n        rows.setdefault(key, r)\n    return [rows[k] for k in sorted(rows)]\n\n\ndef mean(xs):\n    xs = [x for x in xs if x is not None]\n    return sum(xs) / len(xs) if xs else None\n\n\ndef tables(rows):\n    groups = defaultdict(list)\n    for r in rows:\n        groups[(r["config"], r["n_views"])].append(r)\n    means = {}\n    for (c, v), rs in sorted(groups.items()):\n        means[(c, v)] = {"config": c, "n_views": v, "n_scenes": len(rs),\n                         "scenes": " ".join(sorted(r["scene"] for r in rs)),\n                         **{m: mean([r.get(m) for r in rs]) for m in METRICS}}\n    effects = []\n    views = sorted({v for _, v in means})\n    for name, a, b in EFFECTS:\n        for v in views:\n            if (a, v) in means and (b, v) in means:\n                # compare only scenes present in BOTH configs, so a missing run cannot bias the difference\n                sa = {r["scene"]: r for r in groups[(a, v)]}\n                sb = {r["scene"]: r for r in groups[(b, v)]}\n                common = sorted(set(sa) & set(sb))\n                if not common:\n                    continue\n                row = {"effect": name, "n_views": v, "n_scenes": len(common), "scenes": " ".join(common)}\n                for m in ["psnr", "ssim", "lpips", "rot_deg", "rpe_deg", "ate"]:\n                    d = [sa[s][m] - sb[s][m] for s in common if sa[s].get(m) is not None and sb[s].get(m) is not None]\n                    row[f"d_{m}"] = mean(d)\n                    if m == "psnr":\n                        # spread over scenes and how many scenes agree on the sign: a mean alone can hide noise\n                        row["sd_psnr"] = (sum((x - row["d_psnr"]) ** 2 for x in d) / (len(d) - 1)) ** 0.5 if len(d) > 1 else None\n                        row["scenes_positive_psnr"] = f"{sum(x > 0 for x in d)}/{len(d)}"\n                effects.append(row)\n    return list(means.values()), effects\n\n\ndef write_csv(path, rows, fields=None):\n    if not rows:\n        return\n    fields = fields or list(rows[0].keys())\n    with open(path, "w", newline="") as f:\n        w = csv.DictWriter(f, fieldnames=fields, extrasaction="ignore")\n        w.writeheader()\n        for r in rows:\n            w.writerow({k: (round(v, 4) if isinstance(v, float) else v) for k, v in r.items()})\n\n\ndef main(results_root):\n    rows = load(results_root)\n    means, effects = tables(rows)\n    out = Path(results_root)\n    write_csv(out / "all_runs.csv", rows, ["config", "scene", "n_views"] + METRICS + ["date"])\n    write_csv(out / "means_by_config.csv", means)\n    write_csv(out / "effects.csv", effects)\n    print(f"{len(rows)} runs -> all_runs.csv, means_by_config.csv, effects.csv in {out}")\n    for m in means:\n        print(f"  {m[\'config\']} {m[\'n_views\']} photos ({m[\'n_scenes\']} scenes): PSNR {m[\'psnr\']:.2f}  "\n              f"ATE {m[\'ate\']:.3f}  rot {m[\'rot_deg\']:.2f} deg")\n    for e in effects:\n        sd = f", sd {e[\'sd_psnr\']:.2f}" if e.get("sd_psnr") is not None else ""\n        print(f"  {e[\'effect\']} at {e[\'n_views\']} photos ({e[\'n_scenes\']} scenes): dPSNR {e[\'d_psnr\']:+.2f}{sd}, "\n              f"positive in {e[\'scenes_positive_psnr\']} scenes")\n    return rows, means, effects\n\n\ndef selftest():\n    with tempfile.TemporaryDirectory() as d:\n        def put(c, s, v, psnr, ate=0.1, rot=1.0, pm="pose_error.py"):\n            p = Path(d) / c / s / f"{v}_views"\n            p.mkdir(parents=True)\n            json.dump(dict(config=c, scene=s, n_views=v, psnr=psnr, ssim=0.3, lpips=0.5, ate=ate, rot_deg=rot,\n                           rpe_deg=1, ate_init=ate * 2, rot_deg_init=rot * 2, total_seconds=600, peak_vram_mb=4000,\n                           pose_metric=pm, pose_error_version=2, date="x"), open(p / "result.json", "w"))\n        for s, base in [("garden", 15.0), ("bonsai", 14.0)]:\n            put("C1", s, 3, base + 2.0, ate=0.0, rot=0.0)\n            put("C2", s, 3, base)\n            put("C3", s, 3, base + 0.5)\n            put("C4", s, 3, base - 1.0)\n        put("C3_baseline", "garden", 3, 99.0)             # a real C3 run exists -> the baseline copy is ignored\n        put("C3_baseline", "kitchen", 3, 13.0)            # no C3 run -> counts as C3\n        put("C2", "kitchen", 3, 12.0)\n        put("C3", "room", 3, 50.0, pm=None)               # old format, must be ignored\n        put("C4", "stump", 3, 10.0)                       # only in C4: must not enter the C3-C4 effect\n        put("C3_baseline_OLD_interp_init", "bonsai", 3, 1.0)   # renamed old folder: must never be used\n        rows, means, effects = main(d)\n        assert len(rows) == 11, len(rows)\n        assert not any(r["psnr"] == 99.0 for r in rows), "baseline copy must not replace a real C3 run"\n        assert not any(r["psnr"] == 1.0 for r in rows), "renamed old folder must be ignored"\n        e = {x["effect"]: x for x in effects}\n        r3 = e["RQ3 refinement (C3 - C2)"]\n        assert r3["n_scenes"] == 3 and abs(r3["d_psnr"] - (0.5 + 0.5 + 1.0) / 3) < 1e-9, r3\n        e = {x["effect"]: x for x in effects}\n        assert abs(e["RQ1 poses (C1 - C2)"]["d_psnr"] - 2.0) < 1e-9\n        assert abs(e["RQ2 3D points (C3 - C4)"]["d_psnr"] - 1.5) < 1e-9 and e["RQ2 3D points (C3 - C4)"]["n_scenes"] == 2\n        assert (Path(d) / "effects.csv").exists()\n    print("selftest OK - aggregate ignores old/baseline folders and compares only matching scenes")\n\n\nif __name__ == "__main__":\n    if len(sys.argv) > 1 and sys.argv[1] == "--selftest":\n        selftest()\n    else:\n        main(sys.argv[1])\n', 'colab_prepare.py': '"""Shared Colab preparation for every experiment notebook (02, 03, ...).\n\nNotebook 01 must have been run once on the account: it builds the GPU programs and saves them,\nthe MASt3R model, setup_colab.sh and the dataset copy to Drive. This module only restores them.\n\n    import colab_prepare as cp\n    gpu = cp.check_gpu()\n    commit = cp.install(DRIVE, REPO)\n    cp.restore_data(DRIVE, DATA, SCENES, RES_DIR)\n    cp.patch_instantsplat(REPO)\n    manifest = cp.build_manifest(DATA, SCENES, VIEWS, RES_DIR, TEST_STRIDE, DRIVE)\n\nThe two InstantSplat patches below are the same as in notebook 01 (same markers, so a patch is\nnever applied twice):\n  1. Free360-style split: test = every 8th photo; train = n photos spaced evenly among the rest.\n  2. Aligned test-pose start: each test camera starts from its reference pose mapped into the\n     estimated frame with one Sim(3) fitted on the training cameras (InstantSplat\'s default start\n     was 80-90 degrees off in 360-degree scenes).\n"""\nimport json\nimport os\nimport numpy as np\nimport re\nimport struct\nimport subprocess\nfrom pathlib import Path\n\n\ndef sh(cmd, cwd=None):\n    p = subprocess.run(["bash", "-c", cmd], cwd=cwd, capture_output=True, text=True)\n    if p.returncode != 0:\n        raise RuntimeError(f"Command failed: {cmd}\\n{(p.stdout + p.stderr)[-3000:]}")\n    return p.stdout\n\n\n# ----------------------------------------------------------------- GPU, install, data\ndef check_gpu():\n    r = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],\n                       capture_output=True, text=True)\n    if r.returncode != 0 or not r.stdout.strip():\n        raise SystemExit("NO GPU. Runtime -> Change runtime type -> T4 GPU, then Run all again.")\n    name = r.stdout.strip()\n    if "T4" not in name:\n        print("WARNING: not a T4. All FINAL results must come from the same GPU type.")\n    return name\n\n\nINSTANTSPLAT_COMMIT = "b951567"   # every run of the study uses this exact version\n\n\ndef pin_commit(repo, commit=INSTANTSPLAT_COMMIT):\n    """Make sure the cloned InstantSplat is exactly the pinned version (GitHub may move on)."""\n    head = sh(f"cd {repo} && git rev-parse --short HEAD").strip()\n    if not head.startswith(commit[:7]):\n        sh(f"cd {repo} && git fetch -q origin && git checkout -q {commit} && git submodule update -q --init --recursive")\n        head = sh(f"cd {repo} && git rev-parse --short HEAD").strip()\n    assert head.startswith(commit[:7]), f"InstantSplat is at {head}, expected {commit}"\n    return head\n\n\ndef install(drive, repo):\n    """Run the setup script saved by notebook 01 (cached GPU programs -> about 1-3 minutes)."""\n    quick = subprocess.run(["python", "-c", "import diff_gaussian_rasterization, simple_knn, fused_ssim, open3d"],\n                           capture_output=True, text=True)\n    if not (os.path.isdir(repo) and quick.returncode == 0):\n        setup = f"{drive}/setup_colab.sh"\n        if not os.path.exists(setup):\n            raise SystemExit("setup_colab.sh is not on Drive. Run notebook 01 once on this account first.")\n        print(sh(f"bash {setup}")[-300:])\n    return pin_commit(repo)\n\n\ndef restore_data(drive, data, scenes, res_dir="images_8"):\n    """Unpack the dataset copy that notebook 01 saved to Drive and link images -> images_8."""\n    def ready(s):\n        return Path(f"{data}/{s}/{res_dir}").is_dir() and Path(f"{data}/{s}/sparse/0/images.bin").exists()\n\n    tar = f"{drive}/data/mipnerf360_8scenes_{res_dir}.tar"\n    if not all(ready(s) for s in scenes):\n        if not os.path.exists(tar):\n            raise SystemExit(f"Dataset copy not found on Drive ({tar}). Run notebook 01 (version 2) once first.")\n        os.makedirs(data, exist_ok=True)\n        sh(f"tar -xf \'{tar}\' -C {data}")\n    counts = {}\n    for s in scenes:\n        assert ready(s), f"scene {s} missing after unpacking"\n        link = Path(f"{data}/{s}/images")\n        if not link.exists():\n            os.symlink(res_dir, link)\n        counts[s] = len(os.listdir(f"{data}/{s}/{res_dir}"))\n    return counts\n\n\n# ----------------------------------------------------------------- InstantSplat patches (same as notebook 01)\nSPLIT_SRC = \'\'\'def split_train_test(image_files, llffhold=8, n_views=None, verbose=True):\n    # FREE360-STYLE SPLIT (patched by our notebook):\n    # test = every llffhold-th image (idx % llffhold == 0); train = n_views images spaced evenly among the rest\n    n = len(image_files)\n    test_idx = [i for i in range(n) if i % llffhold == 0]\n    train_pool = [i for i in range(n) if i % llffhold != 0]\n    sparse_idx = np.linspace(0, len(train_pool) - 1, num=n_views, dtype=int)\n    train_idx = [train_pool[i] for i in sparse_idx]\n    if verbose:\n        print(">> Spliting Train-Test Set (Free360-style): ")\n        print(" - train_set_indices:  ", train_idx)\n        print(" - test_set_indices:   ", test_idx)\n    train_img_files = [image_files[i] for i in train_idx]\n    test_img_files = [image_files[i] for i in test_idx]\n    return train_img_files, test_img_files\n\'\'\'\n\nINIT_FN = \'\'\'\ndef aligned_test_init(source_path, train_files, test_files, train_w2c):\n    # ALIGNED TEST-POSE INIT: Sim(3) (Umeyama) from reference to estimated training camera centres,\n    # then the reference test poses are mapped into the estimated frame. Same rule for every configuration.\n    import numpy as np, os\n    from scene.colmap_loader import read_extrinsics_binary, qvec2rotmat\n    gt = {os.path.basename(v.name): v for v in read_extrinsics_binary(os.path.join(source_path, "sparse/0/images.bin")).values()}\n    def gt_w2c(f):\n        e = gt[os.path.basename(f)]\n        M = np.eye(4); M[:3, :3] = qvec2rotmat(e.qvec); M[:3, 3] = e.tvec\n        return M\n    est_c2w = np.linalg.inv(np.asarray(train_w2c))\n    gt_c2w = np.stack([np.linalg.inv(gt_w2c(f)) for f in train_files])\n    X, Y = gt_c2w[:, :3, 3], est_c2w[:, :3, 3]\n    mx, my = X.mean(0), Y.mean(0); Xc, Yc = X - mx, Y - my\n    U, D, Vt = np.linalg.svd(Yc.T @ Xc / len(X)); S = np.eye(3)\n    if np.linalg.det(U) * np.linalg.det(Vt) < 0:\n        S[2, 2] = -1\n    R = U @ S @ Vt; s = np.trace(np.diag(D) @ S) / Xc.var(0).sum(); t = my - s * R @ mx\n    out = []\n    for f in test_files:\n        g = np.linalg.inv(gt_w2c(f)); c2w = np.eye(4)\n        c2w[:3, :3] = R @ g[:3, :3]; c2w[:3, 3] = s * R @ g[:3, 3] + t\n        out.append(np.linalg.inv(c2w))\n    return np.stack(out)\n\'\'\'\n\n\ndef patch_instantsplat(repo):\n    sfm = Path(f"{repo}/utils/sfm_utils.py")\n    text = sfm.read_text()\n    if "FREE360-STYLE SPLIT" not in text:\n        new, n = re.subn(r"def split_train_test\\(.*?\\n(?=def get_sorted_image_files)", lambda m: SPLIT_SRC + "\\n\\n",\n                         text, flags=re.S)\n        assert n == 1, "split_train_test not found (InstantSplat changed?)"\n        compile(new, "sfm_utils.py", "exec")\n        sfm.write_text(new)\n        print("patched: Free360-style split")\n    ig = Path(f"{repo}/init_geo.py")\n    text = ig.read_text()\n    if "ALIGNED TEST-POSE INIT" not in text:\n        anchor = "        save_extrinsic(sparse_1_path, pose_test_init, test_img_files, image_suffix)"\n        assert text.count(anchor) == 1 and text.count("\\ndef main(") == 1, "init_geo.py changed - check the patch"\n        text = text.replace(anchor, "        pose_test_init = aligned_test_init(source_path, train_img_files, test_img_files, extrinsics_w2c)\\n" + anchor)\n        text = text.replace("\\ndef main(", INIT_FN + "\\n\\ndef main(", 1)\n        compile(text, "init_geo.py", "exec")\n        ig.write_text(text)\n        print("patched: aligned test-pose start")\n    return True\n\n\n# ----------------------------------------------------------------- exact photo lists\ndef colmap_image_names(path):\n    names = []\n    with open(path, "rb") as f:\n        (n,) = struct.unpack("<Q", f.read(8))\n        for _ in range(n):\n            f.read(64)\n            name = b""\n            while (c := f.read(1)) != b"\\x00":\n                name += c\n            (npts,) = struct.unpack("<Q", f.read(8))\n            f.seek(24 * npts, 1)\n            names.append(name.decode())\n    return names\n\n\ndef instantsplat_order(folder):\n    files = [p for p in Path(folder).iterdir() if p.suffix.lower() in {".png", ".jpg", ".jpeg", ".bmp", ".tiff"}]\n\n    def num(p):\n        m = re.search(r"\\d+", p.stem)\n        return int(m.group()) if m else 10 ** 18\n    return [p.name for p in sorted(files, key=num)]\n\n\ndef build_manifest(data, scenes, views, res_dir="images_8", test_stride=8, drive=None):\n    import numpy as np                                    # used by SPLIT_SRC\n    ns = {"np": np}\n    exec(SPLIT_SRC, ns)\n    split = ns["split_train_test"]\n    manifest = {}\n    for s in scenes:\n        ours = instantsplat_order(f"{data}/{s}/{res_dir}")\n        colmap = sorted(colmap_image_names(f"{data}/{s}/sparse/0/images.bin"))\n        assert ours == colmap, f"{s}: photo order differs between InstantSplat and the reference poses"\n        manifest[s] = {"n_images": len(ours), "test_images": split(ours, test_stride, 3, verbose=False)[1],\n                       "train_images": {str(v): split(ours, test_stride, v, verbose=False)[0] for v in views}}\n    if drive:\n        saved = f"{drive}/views_manifest.json"\n        if os.path.exists(saved):                         # the photo lists must never change between notebooks\n            old = json.load(open(saved))\n            for s in scenes:\n                if s in old:\n                    assert old[s]["test_images"] == manifest[s]["test_images"], f"{s}: test photos changed!"\n                    for v in views:\n                        if str(v) in old[s]["train_images"]:\n                            assert old[s]["train_images"][str(v)] == manifest[s]["train_images"][str(v)], \\\n                                f"{s} {v} views: training photos changed!"\n    return manifest\n\n\n# ----------------------------------------------------------------- where the training cameras are\ndef camera_coverage(data, manifest, scenes, views, drive=None):\n    """For each scene x photo count: how the training cameras are spread around the scene.\n    span_deg  = largest arc covered around the scene (360 = all sides); a small value means one-sided views\n    flatness  = 2nd / 1st singular value of the camera positions (0 = on a line; low values make 3-camera\n                alignment less reliable)\n    The photo choice is InstantSplat\'s rule (evenly spaced by photo number). Several outdoor captures go\n    around the object many times, so this can give one-sided sets (e.g. bicycle 3 photos)."""\n    from pose_error import read_images_bin, w2c_to_cam\n    out = {}\n    for s in scenes:\n        ref = read_images_bin(f"{data}/{s}/sparse/0/images.bin")\n        allc = np.array([w2c_to_cam(*ref[n])[1] for n in sorted(ref)])\n        mu = allc.mean(0)\n        _, _, vt = np.linalg.svd(allc - mu)\n        az = lambda c: float(np.degrees(np.arctan2((c - mu) @ vt[1], (c - mu) @ vt[0])) % 360)\n        out[s] = {}\n        for v in views:\n            names = manifest[s]["train_images"][str(v)]\n            c = np.array([w2c_to_cam(*ref[n])[1] for n in names])\n            a = sorted(az(x) for x in c)\n            gaps = [(a[(i + 1) % len(a)] - a[i]) % 360 for i in range(len(a))]\n            sv = np.linalg.svd(c - c.mean(0), compute_uv=False)\n            out[s][str(v)] = {"azimuths_deg": [round(x) for x in a], "span_deg": round(360 - max(gaps)),\n                              "flatness": round(float(sv[1] / sv[0]), 3)}\n    if drive:\n        json.dump(out, open(f"{drive}/views_coverage.json", "w"), indent=1)\n    return out\n', 'recompute_pose_error.py': '"""Recompute the pose error of finished runs with the current pose_error.py (CPU only, no retraining).\n\nEvery run saves its training poses (results/<config>/<scene>/<n>_views/poses/pose_org.npy and\npose_optimized.npy), so a better pose-error measure can be applied later without using the GPU.\nOld values are kept in result.json under "pose_v<old version>".\n\n    import recompute_pose_error as rp\n    rp.recompute_all(DRIVE + "/results", DATA)\n"""\nimport json\nfrom pathlib import Path\n\nimport pose_error\n\nKEYS = ["ate", "rot_deg", "rpe_t", "rpe_deg"]\n\n\ndef recompute_run(run_dir, data):\n    run_dir = Path(run_dir)\n    r = json.load(open(run_dir / "result.json"))\n    old_version = r.get("pose_error_version", 1)\n    if r.get("pose_metric") != "pose_error.py" or old_version >= pose_error.VERSION:\n        return None\n    if not (run_dir / "poses" / "pose_optimized.npy").exists():\n        return None\n    gt = f"{data}/{r[\'scene\']}/sparse/0/images.bin"\n    names = r["train_images"]\n    after = pose_error.evaluate_run(gt, run_dir / "poses" / "pose_optimized.npy", names)\n    before = pose_error.evaluate_run(gt, run_dir / "poses" / "pose_org.npy", names)\n    r[f"pose_v{old_version}"] = {k: r.get(k) for k in KEYS + [k + "_init" for k in KEYS]}\n    r.update({k: after[k] for k in KEYS})\n    r.update({k + "_init": before[k] for k in KEYS})\n    r["flatness"] = after["flatness"]\n    r["pose_error_version"] = pose_error.VERSION\n    json.dump(r, open(run_dir / "result.json", "w"), indent=1)\n    return r\n\n\ndef recompute_all(results_root, data):\n    changed = []\n    for f in sorted(Path(results_root).glob("*/*/*_views/result.json")):\n        r = recompute_run(f.parent, data)\n        if r:\n            old = r[[k for k in r if k.startswith("pose_v")][0]]\n            print(f"{f.parent.relative_to(results_root)}: rot {old[\'rot_deg\']:.2f} -> {r[\'rot_deg\']:.2f} deg, "\n                  f"ATE {old[\'ate\']:.3f} -> {r[\'ate\']:.3f}, flatness {r[\'flatness\']:.2f}")\n            changed.append(r)\n    print(f"{len(changed)} runs updated to pose error version {pose_error.VERSION}")\n    return changed\n'}

import os, sys, importlib
from pathlib import Path
os.makedirs("/content/scripts", exist_ok=True)
for name, src in SCRIPTS.items():
    Path(f"/content/scripts/{name}").write_text(src)
sys.path.insert(0, "/content/scripts")
import pose_error, run_all, aggregate, colab_prepare
for m in (pose_error, run_all, aggregate, colab_prepare):
    importlib.reload(m)
pose_error.selftest()
aggregate.selftest()
print("OK - scripts ready:", ", ".join(SCRIPTS))

## Step 1 - Setup
**What:** GPU check, Google Drive, install (from the files notebook 01 saved), dataset, our two InstantSplat patches (Free360 split + aligned test-camera start), and the exact photo lists.
**Why:** every notebook must use exactly the same code, data and photos; the photo lists are checked against the ones saved by notebook 01.

In [ ]:
import json
from google.colab import drive
GPU_NAME = "CPU (maintenance)" if CPU_MAINTENANCE else colab_prepare.check_gpu()
print("GPU:", GPU_NAME)
drive.mount("/content/drive")
assert os.path.isdir(DRIVE), f"{DRIVE} not found - see 'Before the first use' at the top."
os.makedirs(f"{DRIVE}/scripts", exist_ok=True)
for name, src in SCRIPTS.items():
    Path(f"{DRIVE}/scripts/{name}").write_text(src)

COMMIT = colab_prepare.INSTANTSPLAT_COMMIT if CPU_MAINTENANCE else colab_prepare.install(DRIVE, REPO)
counts = colab_prepare.restore_data(DRIVE, DATA, ALL_SCENES, RES_DIR)
if not CPU_MAINTENANCE:
    colab_prepare.patch_instantsplat(REPO)
manifest = colab_prepare.build_manifest(DATA, ALL_SCENES, [3, 6, 9], RES_DIR, TEST_STRIDE, DRIVE)

# keep finished runs up to date with the current pose-error version (CPU only, from the saved poses)
import recompute_pose_error
recompute_pose_error.recompute_all(f"{DRIVE}/results", DATA)
# where the training cameras are (arc covered around the scene, flatness) -> Drive/views_coverage.json
coverage = colab_prepare.camera_coverage(DATA, manifest, ALL_SCENES, [3, 6, 9], DRIVE)
print("arc covered by the 3 photos (deg):", {s: coverage[s]["3"]["span_deg"] for s in ALL_SCENES})

ctx = run_all.Context(drive=DRIVE, repo=REPO, data=DATA, local_out=LOCAL_OUT, manifest=manifest,
                      commit=COMMIT, gpu=GPU_NAME, res_dir=RES_DIR, test_stride=TEST_STRIDE)
if CPU_MAINTENANCE:
    CONFIGS_TO_RUN = []        # nothing is trained in maintenance mode
print("OK - setup ready | InstantSplat", COMMIT, "| photos per scene:", counts)

## Step 2 - Switches (Block 2)
**What:** the code that changes one part of the pipeline. It runs right after the AI step (MASt3R) and before training, on the local copy only (the saved MASt3R output on Drive is never changed).
- **C2** needs no code: training without `--optim_pose` keeps the poses fixed.
- **C1 - reference poses:**
  - Map the dataset's reference poses INTO the MASt3R frame. Use one Sim(3) fitted from the reference to the MASt3R training cameras, with `pose_error.umeyama(pose_error.frame_points(...), ...)`. Keep the MASt3R points, the scale and the frame unchanged. Some InstantSplat settings are not scale-free, e.g. the test-pose learning rate and znear/zfar.
  - Write the new training poses to `sparse_<n>/0/images.txt` **and** `images.bin`. The loader reads the `.txt` files.
  - Write the mapped reference **test** poses to `sparse_<n>/1` (the test-camera start).
  - Keep the MASt3R focal length. Note it in `switch_info`.
  - Known limitation to report: with reference poses, the test-camera start is exact, so C1 is also easier to evaluate.
- **C4 - random points:**
  - Use the same number of points, uniform in the box around (points + camera centres), with seed `ctx.seed` and random colours.
  - `confidence_dsp.npy` gives one learning-rate weight per point. Set it to a constant (its mean) for C4, because random points have no MASt3R confidence.
  - Return `{"n_points": ..., "bbox_min": ..., "bbox_max": ..., "confidence": "constant"}`.

Useful facts (checked in the InstantSplat code): training reads the starting points from `DATA/<scene>/sparse_<n>/0/points3D.ply` and the training poses from `DATA/<scene>/sparse_<n>/0/images.bin|txt`; the test-camera start is in `sparse_<n>/1`. `confidence_dsp.npy` in the same folder has one value per point, so keep its length equal to the number of points.

Each function returns a small dict that is saved in `result.json` (for example the number of points and the box).

In [ ]:
def use_reference_poses(ctx, scene, n_views, model):
    # C1 - write the switch here (tracker M3.1), then delete the ".not_ready" line below
    raise NotImplementedError("C1 switch not written yet (tracker M3.1)")
use_reference_poses.not_ready = True


def use_random_points(ctx, scene, n_views, model):
    # C4 - write the switch here (tracker M2.1), then delete the ".not_ready" line below
    raise NotImplementedError("C4 switch not written yet (tracker M2.1)")
use_random_points.not_ready = True


run_all.CONFIGS["C1"]["after_init"] = use_reference_poses
run_all.CONFIGS["C4"]["after_init"] = use_random_points
print("Switches registered:", {k: v["after_init"].__name__ for k, v in run_all.CONFIGS.items()})

## Step 3 - Plan
**What:** lists the runs from the settings, shows which are already finished, and estimates the time.

In [ ]:
todo, finished = [], []
for c in CONFIGS_TO_RUN:
    for s in SCENES:
        for v in VIEWS:
            done = run_all.is_done(run_all.result_dir(ctx, c, s, v)) or (
                c in ctx.reuse and run_all.is_done(run_all.result_dir(ctx, ctx.reuse[c], s, v)))
            (finished if done and not FORCE else todo).append((c, s, v))
minutes = {3: 9, 6: 14, 9: 19}
est = sum(minutes[v] for _, _, v in todo)
print(f"{len(finished)} already finished, {len(todo)} to run (about {est // 60} h {est % 60} min on a T4):")
for t in todo:
    print("  ", *t)

## Step 4 - Run
**What:** runs every planned experiment and saves it to `Drive/results/<config>/<scene>/<n>_views/` (result.json, logs, test images, camera poses). A failed run is reported and the next one starts.

In [ ]:
if CONFIGS_TO_RUN:
    done_runs, failed_runs = run_all.run_batch(ctx, CONFIGS_TO_RUN, SCENES, VIEWS, force=FORCE, dry_run=DRY_RUN)
else:
    print("Nothing to run (CONFIGS_TO_RUN is empty or CPU_MAINTENANCE = True).")

## Step 5 - Checks
**What:** simple tests that each switch did what it should.
- C2 (refinement off): training poses after training must equal the poses before training.
- C3 (refinement on): poses must change during training.
- C1 / C4: the information returned by the switch is shown.

In [ ]:
import numpy as np
for c in CONFIGS_TO_RUN:
    for s in SCENES:
        for v in VIEWS:
            d = Path(run_all.result_dir(ctx, c, s, v))
            if not (d / "result.json").exists():
                continue
            r = json.load(open(d / "result.json"))
            msg = f"{c}/{s}/{v}: PSNR {r['psnr']:.2f}"
            p0, p1 = d / "poses/pose_org.npy", d / "poses/pose_optimized.npy"
            if p0.exists() and p1.exists():
                moved = float(np.abs(np.load(p1) - np.load(p0)).max())
                ok = (moved < 1e-6) if not run_all.CONFIGS[c]["optim_pose"] else (moved > 1e-6)
                msg += f" | poses changed by {moved:.2e} -> {'OK' if ok else 'CHECK!'}"
            if r.get("switch_info"):
                msg += f" | switch: {r['switch_info']}"
            print(msg)

## Step 6 - Tables
**What:** collects every finished run into `results/all_runs.csv`, `means_by_config.csv` and `effects.csv` (the three research questions as differences: C1-C2 poses, C3-C4 points, C3-C2 refinement; only scenes present in both configs are compared; `sd_psnr` and `scenes_positive_psnr` show how consistent an effect is across scenes).

In [ ]:
import pandas as pd
aggregate.main(f"{DRIVE}/results")
for f in ["means_by_config.csv", "effects.csv"]:
    p = f"{DRIVE}/results/{f}"
    if os.path.exists(p):
        print("\n" + f)
        display(pd.read_csv(p).round(3))

## What is saved
- `Drive/results/<config>/<scene>/<n>_views/` - one folder per run
- `Drive/init_cache/<scene>/<n>_views/` - saved MASt3R output, reused by all configurations
- `Drive/results/*.csv` - the tables above
- Write the important numbers and any problem in `LOG.md` and update the live tracker.